# 01 · EDA — dataset

Goal: know the class balance, box sizes and camera conditions **before** training.

> Rule: functions used 2+ times move to `src/vehicle_detect/`. This notebook never writes to `data/processed/` — run `make prepare` / `dvc repro` instead.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

# run from the project root so configs/ and data/ paths work from any notebook
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from vehicle_detect.config import load_config

cfg = load_config("configs/data/thai_traffic.yaml")
classes, sources = cfg["classes"], list(cfg["source_mappings"])
PROC = Path(cfg["paths"]["processed"]) / "coco"


def load_split(name):
    """One row per box (image_id, class, w, h, ...) + one row per image."""
    d = json.loads((PROC / name / "_annotations.coco.json").read_text())
    imgs = pd.DataFrame(d["images"]).rename(columns={"id": "image_id"})
    # processed file names are "<source>_<original name>"
    imgs["source"] = imgs.file_name.map(lambda f: next(s for s in sources if f.startswith(s + "_")))
    imgs["split"] = name
    boxes = pd.DataFrame(d["annotations"])
    boxes["cls"] = boxes.category_id.map({c["id"]: c["name"] for c in d["categories"]})
    boxes[["x", "y", "w", "h"]] = pd.DataFrame(boxes.bbox.tolist(), index=boxes.index)
    return imgs, boxes.merge(imgs[["image_id", "source", "width", "height"]], on="image_id")


parts = {s: load_split(s) for s in ("train", "valid", "test")}
images = pd.concat([p[0] for p in parts.values()], ignore_index=True)
boxes = pd.concat([p[1].assign(split=s) for s, p in parts.items()], ignore_index=True)
print(len(images), "images,", len(boxes), "boxes")
images.groupby(["split", "source"]).size().unstack(fill_value=0)

## 0 · Integrity: can every listed image be decoded?

In [ ]:
def decodes(r):
    try:
        with Image.open(PROC / r.split / r.file_name) as im:
            im.load()
        return True
    except OSError:
        return False


images["ok"] = [decodes(r) for r in images.itertuples()]
broken = images[~images.ok]
print("unreadable images:", len(broken))
broken[["split", "source", "file_name"]]

## 1 · Class balance per split and per source

In [ ]:
bal = boxes.groupby(["cls", "split"]).size().unstack(fill_value=0).reindex(classes, fill_value=0)
bal["train_share_%"] = (100 * bal.train / bal.train.sum()).round(1)
display(bal)
bal[["train", "valid", "test"]].plot.barh(figsize=(8, 4), title="Boxes per class")
plt.gca().set_xscale("symlog")
boxes.groupby(["cls", "source"]).size().unstack(fill_value=0).reindex(classes, fill_value=0)

## 2 · Box size (COCO small < 32², medium < 96², large)

In [ ]:
boxes["area"] = boxes.w * boxes.h
boxes["size"] = pd.cut(boxes.area, [0, 32**2, 96**2, np.inf], labels=["small", "medium", "large"])
size_tbl = pd.crosstab(boxes.cls, boxes["size"], normalize="index").mul(100).round(1)
display(size_tbl.reindex(classes).dropna(how="all"))
print(f"overall small share: {100 * (boxes['size'] == 'small').mean():.1f}%")
boxes.assign(rel=np.sqrt(boxes.area / (boxes.width * boxes.height))).boxplot(
    column="rel", by="cls", rot=45, figsize=(8, 4)
)
plt.suptitle("")
plt.title("sqrt(box area / image area) per class")

## 3 · Density and image sizes

In [ ]:
per_img = boxes.groupby(["split", "image_id"]).size()
print(
    f"boxes/image: mean {per_img.mean():.1f}, p95 {per_img.quantile(0.95):.0f}, max {per_img.max()}"
)
print("images without any box:", len(images) - len(per_img))
images.groupby("source")[["width", "height"]].agg(["min", "median", "max"])

## 4 · Day / night proxy (mean luminance of 400 sampled images)

In [ ]:
rng = np.random.default_rng(0)
good = images[images.ok]
sample = good.iloc[rng.choice(len(good), 400, replace=False)]
lum = []
for r in sample.itertuples():
    im = Image.open(PROC / r.split / r.file_name).convert("L").resize((64, 64))
    lum.append(np.asarray(im).mean())
sample = sample.assign(lum=lum)
sample.lum.hist(bins=30)
plt.title("mean luminance (0 black .. 255 white)")
print(f"dark (<70): {100 * (sample.lum < 70).mean():.1f}%")
sample.groupby("source").lum.describe().round(1)

## 5 · Look at the images (120 random, with boxes)

Outputs are stripped by nbstripout before commit, so no frames reach git (PDPA).

In [ ]:
from PIL import ImageDraw

by_img = boxes.groupby(["split", "image_id"])
look = images[images.ok].sample(120, random_state=1)


def draw(r):
    im = Image.open(PROC / r.split / r.file_name).convert("RGB")
    d = ImageDraw.Draw(im)
    if (r.split, r.image_id) in by_img.groups:
        for b in by_img.get_group((r.split, r.image_id)).itertuples():
            d.rectangle([b.x, b.y, b.x + b.w, b.y + b.h], outline="red", width=3)
    return im.resize((320, 240))


for page in range(4):
    fig, axes = plt.subplots(5, 6, figsize=(18, 11))
    for ax, r in zip(axes.flat, look.iloc[page * 30 : (page + 1) * 30].itertuples(), strict=False):
        ax.imshow(draw(r))
        ax.set_title(r.source, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 6 · Rare classes: tuk_tuk / songthaew

In [ ]:
rare = boxes[boxes.cls.isin(["tuk_tuk", "songthaew"])]
print(rare.groupby(["cls", "split", "source"]).size())
print("distinct images with a rare class:", rare.groupby("cls").image_id.nunique().to_dict())